## Library

In [2]:
import time
from dataretrieval import nwis
import pandas as pd
from IPython.display import clear_output

ERROR! Session/line number was not unique in database. History logging moved to new session 4158


## Download Gage Height (00065) for SWOT Version D

In [3]:
# ============================================================================
# Download USGS gage height data for SWOT observation period
# ============================================================================
from dataretrieval import nwis
import pandas as pd
import time
from IPython.display import clear_output

# SWOT observation period (Version D): from operational start to end of October 2025
START, END = "2023-01-01", "2025-10-31"
STATES = [
    "AL","AK","AZ","AR","CA","CO","CT","DC","DE","FL","GA","HI","IA","ID","IL",
    "IN","KS","KY","LA","MA","MD","ME","MI","MN","MO","MS","MT","NC","ND","NE",
    "NH","NJ","NM","NV","NY","OH","OK","OR","PA","PR","RI","SC","SD","TN","TX",
    "UT","VA","VI","VT","WA","WI","WV","WY"
]

all_parts = []
for st in STATES:
    clear_output(wait=True)
    print(f"Processing {st}...")
    
    try:
        # Download gage height (00065)
        df = nwis.get_record(
            service="dv",
            stateCd=st,
            start=START, 
            end=END,
            parameterCd="00065",
            siteStatus="all",
        )
        
        if not df.empty:
            df["stateCd"] = st
            all_parts.append(df)
            print(f"  → {st}: {len(df)} records downloaded")
        else:
            print(f"  → {st}: No data")
            
    except Exception as e:
        print(f"  → {st}: Error - {str(e)}")
        continue
    
    time.sleep(0.5)

# Concatenate all state data
all_df = pd.concat(all_parts, axis=0, ignore_index=False)
print(f"\nTotal records: {len(all_df)}")

# Check index structure and get site count
if isinstance(all_df.index, pd.MultiIndex):
    print(f"Total sites: {all_df.index.get_level_values('site_no').nunique()}")
else:
    # If index is reset or different structure
    all_df_reset = all_df.reset_index()
    if 'site_no' in all_df_reset.columns:
        print(f"Total sites: {all_df_reset['site_no'].nunique()}")
    else:
        print("Index structure:", all_df.index.names)
        print("Columns:", list(all_df.columns))

# Save to parquet
output_path = "/work/pi_kandread_umass_edu/swot-urban/1_data_processing/usgs_gages/usgs_gage_height_23_25.parquet"
all_df.to_parquet(output_path)
print(f"\nSaved to: {output_path}")

Processing WY...
  → WY: 1732 records downloaded

Total records: 4604885
Total sites: 4


ArrowInvalid: ('cannot mix list and non-list, non-null values', 'Conversion failed for column None with type object')

In [2]:
import pandas as pd

# Load data
df = pd.read_parquet("usgs_dv_Q_V_Stage_swot_period_2023_2025.parquet")

In [5]:
df.columns

Index(['00060_Mean', '00060_Mean_cd', '00065_reservoir elevation_Mean',
       '00065_reservoir elevation_Mean_cd', '00065_Mean', '00065_Mean_cd',
       'stateCd', '00065_dcp_Mean', '00065_dcp_Mean_cd',
       '00060_data from 10/1/1992 forward_Mean',
       ...
       '00060_10_Mean', '00060_10_Mean_cd', '00065_continuous_Mean',
       '00065_continuous_Mean_cd', '00065_mean lake elev, stage datum_Mean',
       '00065_mean lake elev, stage datum_Mean_cd',
       '00065_tailwater (base_Mean', '00065_tailwater (base_Mean_cd',
       '00065_auxiliary gage_Mean', '00065_auxiliary gage_Mean_cd'],
      dtype='object', length=261)

In [6]:
# 00065 (Stage) 체크
stage_cols = [col for col in df.columns if col.startswith('00065_') and '_cd' not in col]
print(f"총 00065 컬럼 개수: {len(stage_cols)}")

# 00065_Mean이 없지만 다른 00065_*에는 값이 있는 경우
mask_no_primary = df['00065_Mean'].isna()
other_stage_cols = [col for col in stage_cols if col != '00065_Mean']

has_alternative = pd.Series(False, index=df.index)
for col in other_stage_cols:
    has_alternative |= df[col].notna()

# 00065_Mean은 없지만 다른 컬럼에는 있는 경우
additional_from_alternatives = (mask_no_primary & has_alternative).sum()

print(f"\n=== Stage (00065) 분석 ===")
print(f"00065_Mean에 값이 있는 행: {df['00065_Mean'].notna().sum():,}")
print(f"00065_Mean은 없지만 다른 00065_*에는 값이 있는 행: {additional_from_alternatives:,}")
print(f"  → 추가로 얻을 수 있는 데이터: {additional_from_alternatives:,}개")

# 어떤 컬럼들이 추가 데이터를 제공하는지 확인
print(f"\n추가 데이터를 제공하는 컬럼들:")
for col in other_stage_cols:
    contributes = (mask_no_primary & df[col].notna()).sum()
    if contributes > 0:
        print(f"  {col}: {contributes:,}개 추가")

# 00060 (Discharge)도 동일하게 체크
print(f"\n{'='*50}")
discharge_cols = [col for col in df.columns if col.startswith('00060_') and '_cd' not in col]
print(f"총 00060 컬럼 개수: {len(discharge_cols)}")

mask_no_primary_q = df['00060_Mean'].isna()
other_discharge_cols = [col for col in discharge_cols if col != '00060_Mean']

has_alternative_q = pd.Series(False, index=df.index)
for col in other_discharge_cols:
    has_alternative_q |= df[col].notna()

additional_from_alternatives_q = (mask_no_primary_q & has_alternative_q).sum()

print(f"\n=== Discharge (00060) 분석 ===")
print(f"00060_Mean에 값이 있는 행: {df['00060_Mean'].notna().sum():,}")
print(f"00060_Mean은 없지만 다른 00060_*에는 값이 있는 행: {additional_from_alternatives_q:,}")
print(f"  → 추가로 얻을 수 있는 데이터: {additional_from_alternatives_q:,}개")

print(f"\n추가 데이터를 제공하는 컬럼들:")
for col in other_discharge_cols:
    contributes = (mask_no_primary_q & df[col].notna()).sum()
    if contributes > 0:
        print(f"  {col}: {contributes:,}개 추가")

총 00065 컬럼 개수: 78

=== Stage (00065) 분석 ===
00065_Mean에 값이 있는 행: 3,644,135
00065_Mean은 없지만 다른 00065_*에는 값이 있는 행: 164,115
  → 추가로 얻을 수 있는 데이터: 164,115개

추가 데이터를 제공하는 컬럼들:
  00065_reservoir elevation_Mean: 970개 추가
  00065_dcp_Mean: 833개 추가
  00065_headwater_Mean: 9,731개 추가
  00065_tailwater_Mean: 10,471개 추가
  00065_upstream_Mean: 2,985개 추가
  00065_to compute gh_Mean: 995개 추가
  00065_ngvd29_Mean: 29,009개 추가
  00065_navd88_Mean: 23,289개 추가
  00065_downstream_Mean: 2,003개 추가
  00065_navd 1988_Mean: 1,003개 추가
  00065_upstream, ngvd29_Mean: 7,016개 추가
  00065_downstream, ngvd29_Mean: 7,941개 추가
  00065_dns ngvd29_Mean: 1,004개 추가
  00065_upstream, [upstream_Mean: 1,353개 추가
  00065_downstream, [downstream_Mean: 1,297개 추가
  00065_usace ups_Mean: 1,003개 추가
  00065_above structure_Mean: 1,004개 추가
  00065_tallahasee records_Mean: 1,004개 추가
  00065_usgs data_Mean: 1,002개 추가
  00065_navd88, [navd88_Mean: 988개 추가
  00065_upstream ngvd29_Mean: 1,985개 추가
  00065_downstream ngvd29_Mean: 1,995개 추가
  00065_n

In [9]:
import pandas as pd
import numpy as np

# 원본 df 사용 (다시 시작)
# 00065_Mean이 NaN인 행만 처리 대상

# ============================================================================
# 1. 조합 패턴 정의
# ============================================================================

# 위치가 다른 경우 → former/later로 분리
spatial_pairs = {
    'upstream_downstream': [
        ('00065_upstream_Mean', '00065_downstream_Mean'),
        ('00065_upstream, ngvd29_Mean', '00065_downstream, ngvd29_Mean'),
        ('00065_upstream ngvd29_Mean', '00065_downstream ngvd29_Mean'),
        ('00065_upstream, [upstream_Mean', '00065_downstream, [downstream_Mean'),
        ('00065_downstream, ngvd29_Mean', '00065_upstream, ngvd29, [upstream_Mean'),
    ],
    'headwater_tailwater': [
        ('00065_headwater_Mean', '00065_tailwater_Mean'),
    ],
    'protected_flood': [
        ('00065_protected side, [protected side_Mean', '00065_flood side, [flood side_Mean'),
    ],
    'outside_inside': [
        ('00065_outside lock chamber_Mean', '00065_inside lock chamber_Mean'),
    ],
    'base_auxiliary': [
        ('00065_base gage_Mean', '00065_auxiliary gage, [auxiliary gage_Mean'),
    ],
    'east_west': [
        ('00065_east gage, ngvd29_Mean', '00065_west gage, ngvd29_Mean'),
    ],
}

# 같은 위치, 다른 센서 → 평균내서 Mean으로
sensor_pairs = [
    ('00065_primary sensor_Mean', '00065_secondary sensor_Mean'),
    ('00065_stage a_Mean', '00065_stage b_Mean'),
    ('00065_ngvd29_Mean', '00065_navd88_Mean'),  # datum 차이도 평균
]

# ============================================================================
# 2. 새로운 컬럼 초기화
# ============================================================================
if '00065_Mean' not in df.columns:
    df['00065_Mean'] = np.nan
    
df['00065_former'] = np.nan  # upstream/headwater/protected 등
df['00065_later'] = np.nan   # downstream/tailwater/flood 등
df['00065_source'] = None    # 어디서 왔는지 추적

# 기존 값이 있는 경우 표시
df.loc[df['00065_Mean'].notna(), '00065_source'] = 'original'

# ============================================================================
# 3. Spatial pairs 처리 (former/later로 분리)
# ============================================================================
print("=== Spatial pairs 처리 (위치가 다른 경우) ===")
spatial_filled = 0

for category, pairs in spatial_pairs.items():
    for former_col, later_col in pairs:
        if former_col not in df.columns or later_col not in df.columns:
            continue
        
        # 00065_Mean이 NaN이고, 두 컬럼 모두 값이 있는 경우
        mask = (df['00065_Mean'].isna() & 
                df[former_col].notna() & 
                df[later_col].notna())
        
        if mask.sum() > 0:
            df.loc[mask, '00065_former'] = df.loc[mask, former_col]
            df.loc[mask, '00065_later'] = df.loc[mask, later_col]
            df.loc[mask, '00065_source'] = f"{former_col} + {later_col}"
            spatial_filled += mask.sum()
            print(f"  {former_col} + {later_col}: {mask.sum():,} 행")

print(f"→ 총 {spatial_filled:,} 행 처리 (former/later 분리)")

# ============================================================================
# 4. Sensor pairs 처리 (평균내서 Mean으로)
# ============================================================================
print(f"\n=== Sensor/Datum pairs 처리 (평균) ===")
sensor_filled = 0

for col1, col2 in sensor_pairs:
    if col1 not in df.columns or col2 not in df.columns:
        continue
    
    # 00065_Mean이 NaN이고, 두 컬럼 모두 값이 있는 경우
    mask = (df['00065_Mean'].isna() & 
            df[col1].notna() & 
            df[col2].notna())
    
    if mask.sum() > 0:
        # 평균 계산
        df.loc[mask, '00065_Mean'] = (df.loc[mask, col1] + df.loc[mask, col2]) / 2
        df.loc[mask, '00065_source'] = f"avg({col1} + {col2})"
        sensor_filled += mask.sum()
        print(f"  avg({col1} + {col2}): {mask.sum():,} 행")

print(f"→ 총 {sensor_filled:,} 행 처리 (평균으로 Mean 채움)")

# ============================================================================
# 5. 결과 확인
# ============================================================================
print(f"\n=== 최종 결과 ===")
print(f"00065_Mean (원본): {(df['00065_source'] == 'original').sum():,} 행")
print(f"00065_Mean (평균으로 채움): {sensor_filled:,} 행")
print(f"00065_former/later (분리): {spatial_filled:,} 행")
print(f"00065_Mean 총 데이터: {df['00065_Mean'].notna().sum():,} 행")
print(f"00065_former 데이터: {df['00065_former'].notna().sum():,} 행")
print(f"00065_later 데이터: {df['00065_later'].notna().sum():,} 행")

# ============================================================================
# 6. 상세 통계
# ============================================================================
print(f"\n=== 데이터 출처 분포 ===")
source_dist = df['00065_source'].value_counts()
print(source_dist.head(20))

# ============================================================================
# 7. 검증: former/later가 있는데 Mean도 있는 경우 (있어선 안 됨)
# ============================================================================
both_exist = (df['00065_Mean'].notna() & 
              (df['00065_former'].notna() | df['00065_later'].notna())).sum()
if both_exist > 0:
    print(f"\n⚠️ 주의: Mean과 former/later가 동시에 있는 행: {both_exist:,} 개")
    print("  → 이는 원래 00065_Mean이 있던 행에 former/later 정보가 추가된 경우일 수 있음")
else:
    print(f"\n✅ Mean과 former/later는 서로 배타적입니다!")

# ============================================================================
# 8. 예시 데이터 확인
# ============================================================================
print(f"\n=== 샘플 데이터 (former/later 분리된 경우) ===")
sample_spatial = df[df['00065_former'].notna()].head(5)
print(sample_spatial[['00065_Mean', '00065_former', '00065_later', '00065_source']])

print(f"\n=== 샘플 데이터 (평균으로 채워진 경우) ===")
sample_avg = df[df['00065_source'].str.contains('avg', na=False)].head(5)
print(sample_avg[['00065_Mean', '00065_source']])

=== Spatial pairs 처리 (위치가 다른 경우) ===
  00065_upstream_Mean + 00065_downstream_Mean: 1,987 행
  00065_upstream, ngvd29_Mean + 00065_downstream, ngvd29_Mean: 6,933 행
  00065_upstream ngvd29_Mean + 00065_downstream ngvd29_Mean: 1,985 행
  00065_upstream, [upstream_Mean + 00065_downstream, [downstream_Mean: 1,275 행
  00065_downstream, ngvd29_Mean + 00065_upstream, ngvd29, [upstream_Mean: 990 행
  00065_headwater_Mean + 00065_tailwater_Mean: 9,150 행
  00065_protected side, [protected side_Mean + 00065_flood side, [flood side_Mean: 3,789 행
  00065_outside lock chamber_Mean + 00065_inside lock chamber_Mean: 2,947 행
  00065_east gage, ngvd29_Mean + 00065_west gage, ngvd29_Mean: 987 행
→ 총 30,043 행 처리 (former/later 분리)

=== Sensor/Datum pairs 처리 (평균) ===
  avg(00065_stage a_Mean + 00065_stage b_Mean): 992 행
→ 총 992 행 처리 (평균으로 Mean 채움)

=== 최종 결과 ===
00065_Mean (원본): 3,757,977 행
00065_Mean (평균으로 채움): 992 행
00065_former/later (분리): 30,043 행
00065_Mean 총 데이터: 3,758,969 행
00065_former 데이터: 30,043 행
000